# Client B trên Wi-Fi

Chạy server trước. Chạy hai ô code đầu một lần, sau đó lặp các ô gửi và đọc. Không Run All vì ô cuối đóng kết nối. Không cần chạy nhanh giữa các ô.


## 1 Kết nối



In [ ]:
import socket
import threading
import queue
HOST = '172.20.10.12'
PORT = 5000
client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
client.settimeout(3)
client.connect((HOST, PORT))
client.settimeout(None)
inbox = queue.Queue()
print('Client:', client.getsockname(), '->', client.getpeername())


## 2 Chuẩn bị gửi và nhận



In [ ]:
def receive_forever():
    try:
        with client.makefile('rb') as reader:
            while True:
                raw = reader.readline(8193)
                if not raw:
                    break
                if len(raw) > 8192 or not raw.endswith(b'\n'):
                    break
                inbox.put(raw[:-1].decode('utf-8'))
    except (OSError, UnicodeError):
        pass
    finally:
        inbox.put('[Kết nối đã đóng]')
receiver = threading.Thread(target=receive_forever, daemon=True)
receiver.start()
def send(text):
    if not text or '\n' in text or '\r' in text:
        raise ValueError('Tin nhắn phải có nội dung và nằm trên một dòng')
    payload = (text + '\n').encode('utf-8')
    if len(payload) > 4096:
        raise ValueError('Tin nhắn quá dài')
    client.sendall(payload)
history = []
def read_messages():
    while True:
        try:
            history.append(inbox.get_nowait())
        except queue.Empty:
            break
    if not history:
        print('Chưa có tin nhắn')
    for message in history:
        print(message)


## 3 Gửi tin



In [ ]:
send('Chào mọi người, tôi là Bình!')


## 4 Xem tin nhận được

In toàn bộ lịch sử đã nhận trong phiên này. Chạy lại ô này khi muốn xem tin.


In [ ]:
read_messages()


## 5 Đóng client

Chỉ chạy khi kết thúc buổi thực hành.


In [ ]:
try:
    client.shutdown(socket.SHUT_RDWR)
except OSError:
    pass
client.close()
receiver.join(2)
print('Đã đóng client')
